In [ ]:
import os
import time
import warnings
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve
)
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

os.environ.setdefault('LOKY_MAX_CPU_COUNT', '1')
warnings.filterwarnings('ignore', message='IProgress not found.*')
warnings.filterwarnings(
    'ignore',
    message='X does not have valid feature names, but LGBMClassifier was fitted with feature names'
)

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
MODEL_ORDER = ['AdaBoost', 'XGBoost', 'LightGBM', 'CatBoost']
from cross_validation_utils import (
    CV_PROTOCOL, split_summary, load_cv_result, summary_row
)


In [ ]:
df = pd.read_csv('sylhet_diabetes.csv')

symptom_columns = [
    'Polyuria', 'Polydipsia', 'sudden weight loss', 'weakness',
    'Polyphagia', 'Genital thrush', 'visual blurring', 'Itching',
    'Irritability', 'delayed healing', 'partial paresis',
    'muscle stiffness', 'Alopecia', 'Obesity'
]
df[symptom_columns] = df[symptom_columns].apply(
    lambda column: column.str.strip().map({'No': 0, 'Yes': 1})
)
df['Gender'] = df['Gender'].str.strip().map({'Female': 0, 'Male': 1})
df['class'] = df['class'].str.strip().map({'Negative': 0, 'Positive': 1})
assert not df.isna().any().any(), 'Ada kategori yang gagal diencoding.'

X = df.drop('class', axis=1)
y = df['class'].astype(int)
groups = X.groupby(list(X.columns), sort=False, dropna=False).ngroup()
profile_label_counts = df.groupby(
    list(X.columns), sort=False, dropna=False
)['class'].nunique()
assert profile_label_counts.max() == 1, 'Ada profil identik dengan label bertentangan.'

N_SPLITS = 5

outer_cv = StratifiedGroupKFold(
    n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE
)

# Seluruh fold dipakai; setiap baris menjadi validation tepat sekali.
split_summary(X, y, outer_cv, groups)


In [ ]:
# Parameter model dibaca otomatis dari JSON, bukan ditulis manual.
import json
from pathlib import Path

params_path = Path('best_params_sylhet.json')
if not params_path.exists():
    raise FileNotFoundError(
        'Jalankan cell hasil tuning pada setiap notebook model terlebih dahulu.'
    )

tuning_results = json.loads(params_path.read_text(encoding='utf-8'))
print(f'Parameters loaded from: {params_path.resolve()}')
missing_models = [name for name in MODEL_ORDER if name not in tuning_results]
if missing_models:
    raise ValueError(f'Hasil tuning belum tersedia untuk: {missing_models}')

best_params = {
    name: tuning_results[name]['params'] for name in MODEL_ORDER
}
cv_best_f1 = {
    name: tuning_results[name]['cv_f1'] for name in MODEL_ORDER
}

models = {
    'AdaBoost': AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=best_params['AdaBoost']['max_depth'],
            random_state=RANDOM_STATE
        ),
        learning_rate=best_params['AdaBoost']['learning_rate'],
        n_estimators=best_params['AdaBoost']['n_estimators'],
        random_state=RANDOM_STATE
    ),
    'XGBoost': XGBClassifier(
        objective='binary:logistic', eval_metric='logloss',
        tree_method='hist', device='cuda', random_state=RANDOM_STATE,
        n_jobs=1, **best_params['XGBoost']
    ),
    'LightGBM': LGBMClassifier(
        objective='binary', boosting_type='gbdt', subsample_freq=1,
        random_state=RANDOM_STATE, n_jobs=1, verbosity=-1,
        **best_params['LightGBM']
    ),
    'CatBoost': CatBoostClassifier(
        loss_function='Logloss', random_seed=RANDOM_STATE, verbose=0,
        allow_writing_files=False, thread_count=1,
        **best_params['CatBoost']
    )
}
assert list(models) == MODEL_ORDER

pd.DataFrame({
    'Model': MODEL_ORDER,
    'Best CV F1': [cv_best_f1[name] for name in MODEL_ORDER],
    'Best Params': [best_params[name] for name in MODEL_ORDER]
})

In [ ]:
cv_results = {}
predictions = {}
probabilities = {}
fit_seconds = {}

# Ambil hasil outer folds langsung dari notebook model, tanpa fit ulang.
for name in MODEL_ORDER:
    result = load_cv_result(tuning_results[name], X, y, outer_cv, groups)
    cv_results[name] = result
    predictions[name] = result['prediction']
    probabilities[name] = result['probability']
    fit_seconds[name] = result['folds']['Fit seconds'].sum()
    print(f"{name}: nested 5-fold CV results loaded from "
          f"{tuning_results[name]['cv_result_file']}")


In [ ]:
# Nama array dipertahankan untuk plot; isinya label seluruh prediksi OOF.
y_test_array = y.to_numpy()

rows = []
for name in MODEL_ORDER:
    rows.append({'Model': name, **summary_row(cv_results[name]),
                 'Tuning CV F1': cv_best_f1[name]})

results_df = pd.DataFrame(rows).reset_index(drop=True)
results_df.to_csv('comparison_Sylhet_results.csv', index=False)
results_df.round(4)


In [ ]:
metrics_to_plot = ['Balanced Accuracy', 'F1', 'ROC-AUC', 'PR-AUC']
plot_df = results_df.melt(
    id_vars='Model', value_vars=metrics_to_plot,
    var_name='Metric', value_name='Score'
)

plt.figure(figsize=(11, 5))
ax = sns.barplot(
    data=plot_df, x='Metric', y='Score', hue='Model', hue_order=MODEL_ORDER
)
ax.set_ylim(0.65, 1.0)
ax.set_title(f'Main Metric Comparison - Sylhet (Outer {N_SPLITS}-Fold CV)')
ax.set_xlabel('Metric')
ax.set_ylabel('Score')
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=8, padding=2)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
classification_metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
classification_plot_df = results_df.melt(
    id_vars='Model', value_vars=classification_metrics,
    var_name='Metric', value_name='Score'
)

plt.figure(figsize=(11, 5))
ax = sns.barplot(
    data=classification_plot_df, x='Metric', y='Score', hue='Model',
    hue_order=MODEL_ORDER
)
ax.set_ylim(0, 1)
ax.set_title('Comparison of Classification Metrics in Sylhet')
ax.set_xlabel('Metric')
ax.set_ylabel('Score')
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=8, padding=2)
plt.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Grafik berikut memakai prediksi OOF gabungan, bukan mean metrik fold.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for name in MODEL_ORDER:
    fpr, tpr, _ = roc_curve(y_test_array, probabilities[name])
    axes[0].plot(
        fpr, tpr,
        label=f'{name} (AUC={roc_auc_score(y_test_array, probabilities[name]):.3f})'
    )
    precision, recall, _ = precision_recall_curve(
        y_test_array, probabilities[name]
    )
    axes[1].plot(
        recall, precision,
        label=f'{name} (AP={average_precision_score(y_test_array, probabilities[name]):.3f})'
    )

axes[0].plot([0, 1], [0, 1], '--', color='gray', label='Random')
axes[0].set(title='ROC Curve', xlabel='False Positive Rate', ylabel='True Positive Rate')
axes[1].axhline(y_test_array.mean(), linestyle='--', color='gray', label='Prevalence')
axes[1].set(title='Precision-Recall Curve', xlabel='Recall', ylabel='Precision')
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Grafik memakai prediksi OOF gabungan, bukan mean metrik per fold.
cm_fold_ids = cv_results[MODEL_ORDER[0]]['fold_id']
print(f'Combined OOF confusion matrix ({N_SPLITS} folds): {len(y_test_array)} samples.')

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, name in zip(axes.ravel(), MODEL_ORDER):
    assert np.array_equal(cv_results[name]['fold_id'], cm_fold_ids), \
        'Seluruh model harus memakai outer folds yang sama.'
    matrix = confusion_matrix(y_test_array, predictions[name])
    sns.heatmap(
        matrix, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
        xticklabels=['Negative', 'Positive'],
        yticklabels=['Negative', 'Positive']
    )
    ax.set_title(f'{name} - OOF {N_SPLITS}-Fold (n={len(y_test_array)})')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Ringkasan deskriptif: OOF saling bergantung karena training folds overlap.
# Karena itu, jangan gunakan p-value McNemar biasa sebagai uji signifikansi.
discordance_rows = []
for model_a, model_b in combinations(MODEL_ORDER, 2):
    correct_a = predictions[model_a] == y_test_array
    correct_b = predictions[model_b] == y_test_array
    discordance_rows.append({
        'Model A': model_a, 'Model B': model_b,
        'A benar, B salah': int(np.sum(correct_a & ~correct_b)),
        'A salah, B benar': int(np.sum(~correct_a & correct_b))
    })
discordance_df = pd.DataFrame(discordance_rows)
discordance_df


In [ ]:
prediction_df = pd.DataFrame({
    'row_index': X.index.to_numpy(),
    'group_id': np.asarray(groups),
    'fold': cv_results[MODEL_ORDER[0]]['fold_id'],
    'actual': y_test_array
})
for name in MODEL_ORDER:
    safe_name = name.lower()
    prediction_df[f'{safe_name}_prediction'] = predictions[name]
    prediction_df[f'{safe_name}_probability'] = probabilities[name]
prediction_df.to_csv('comparison_Sylhet_predictions.csv', index=False)

best_cv_f1 = results_df.loc[results_df['F1'].idxmax(), 'Model']
best_pr_auc = results_df.loc[results_df['PR-AUC'].idxmax(), 'Model']
print(f'Model with the highest mean outer-CV F1: {best_cv_f1}')
print(f'Model with the highest mean outer-CV PR-AUC: {best_pr_auc}')
print('Results saved to comparison_Sylhet_results.csv')
print('OOF predictions saved to comparison_Sylhet_predictions.csv')
